In [1]:
!pip install torch
import subprocess
import sys
import torch

cuda_available = torch.cuda.is_available()
major_version = 0
if cuda_available:
    major_version, _ = torch.cuda.get_device_capability()

def run_pip(args, allow_fail=False):
    cmd = [sys.executable, "-m", "pip", *args]
    result = subprocess.run(cmd, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if result.returncode != 0 and not allow_fail:
        raise RuntimeError(f"pip failed: {' '.join(cmd)}")
    return result.returncode == 0

# Keep pip tooling up to date so wheel resolution is more reliable on Colab.
run_pip(["install", "-U", "pip", "setuptools", "wheel"])

# Install Unsloth first.
run_pip(["install", "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"])

# Install core runtime packages. Avoid flash-attn as a hard dependency.
common_pkgs = ["trl<0.9.0", "peft", "accelerate", "bitsandbytes"]
xformers_ok = run_pip(
    ["install", "--no-deps", "--only-binary=:all:", "xformers", *common_pkgs],
    allow_fail=True,
)
if not xformers_ok:
    print("xformers wheel not available for this runtime; installing without xformers.")
    run_pip(["install", "--no-deps", *common_pkgs])

# flash-attn is optional and often unavailable on Colab wheels; skip to keep logs clean.
if cuda_available and major_version >= 8:
    print("Skipping optional flash-attn install in this notebook setup.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 74.2 MB/s eta 0:00:00
  Attempting uninstall: wheel
    Found existing installation: wheel 0.46.3
    Uninstalling wheel-0.46.3:
      Successfully uninstalled wheel-0.46.3
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.2.0
    Uninstalling setuptools-75.2.0:
      Successfully uninstalled setuptools-75.2.0
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-sa3wvlxh/unsloth_d01de6f01d7b4b3e9f706e7a9bdca9a6
  Resolved https://gith

In [2]:
import re
import nltk
from collections import Counter
from datasets import load_dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm.auto import tqdm

nltk.download('punkt_tab')
from nltk.tokenize import sent_tokenize

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [3]:
import os
import random
import numpy as np

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Make common PyTorch ops deterministic where possible.
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(f"Seeds set to {SEED}")

Seeds set to 42


In [4]:
#Load Existing model using Unsloth and give one example from Mergers and Acquistions News

from unsloth import FastLanguageModel
import torch

if torch.cuda.is_available():
    major_cap, _ = torch.cuda.get_device_capability()
else:
    major_cap = 0

max_seq_length = 3072
dtype = torch.bfloat16 if major_cap >= 8 else torch.float16
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/llama-3-8b-bnb-4bit",  # MODEL USED
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
    gpu_memory_utilization=0.9,
 )
FastLanguageModel.for_inference(model)  # Enable native faster inference

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/tmp/ipykernel_3769/1184303074.py:3: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.8: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

...

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

Unsloth: Will load unsloth/llama-3-8b-bnb-4bit as a legacy tokenizer.


In [5]:
raw_dataset = load_dataset("hotpot_qa", "distractor")
dataset = DatasetDict({
    "validation": raw_dataset["validation"].select(range(200))  # start small
})
val_set = dataset["validation"]

def _extract_titles_and_sentences(example):
    ctx = example["context"]

    # Handles dict-style context: {"title": [...], "sentences": [[...], ...]}
    if isinstance(ctx, dict):
        titles = ctx.get("title", [])
        sent_lists = ctx.get("sentences", ctx.get("sents", []))
        return titles, sent_lists

    # Handles list/tuple style context: [titles, sentences]
    if isinstance(ctx, (list, tuple)) and len(ctx) == 2:
        return ctx[0], ctx[1]

    # Fallback: unknown shape
    return [], []

def build_context(example, max_sentences=50):
    context_lines = []
    count = 0

    titles, sent_lists = _extract_titles_and_sentences(example)

    for title, sents in zip(titles, sent_lists):
        for sent in sents:
            if sent and sent.strip():
                context_lines.append(f"{title}: {sent}")
                count += 1
            if count >= max_sentences:  # prevent huge prompts
                return "\n".join(context_lines)

    return "\n".join(context_lines)

README.md: 0.00B [00:00, ?B/s]

distractor/train-00000-of-00002.parquet:   0%|          | 0.00/166M [00:00<?, ?B/s]

distractor/train-00001-of-00002.parquet:   0%|          | 0.00/166M [00:00<?, ?B/s]

distractor/validation-00000-of-00001.par(…):   0%|          | 0.00/27.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/90447 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/7405 [00:00<?, ? examples/s]

In [6]:
def prompt_baseline(q, c):
    return f"""Answer the question based on the context.

Context:
{c}

Question: {q}
Answer:"""

def prompt_grounded(q, c):
    return f"""Answer using ONLY the context. If unsupported, say "NOT ENOUGH INFORMATION".

Context:
{c}

Question: {q}
Answer:"""

def prompt_cot(q, c):
    return f"""Answer step by step.

Context:
{c}

Question: {q}
Reasoning:
Answer:"""

def prompt_cot_citation(q, c):
    return f"""Answer step by step and cite evidence.

Context:
{c}

Question: {q}
Reasoning (with citations):
Answer:"""

In [7]:
import json

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_MAX_SEQ_LEN = int(max_seq_length) if "max_seq_length" in globals() else 3072

def _coerce_to_text(value):
    if isinstance(value, str):
        return value
    if value is None:
        return ""
    if isinstance(value, (list, tuple)):
        return "\n".join(_coerce_to_text(v) for v in value)
    if isinstance(value, dict):
        return json.dumps(value, ensure_ascii=False)
    return str(value)

def generate_answer(
    prompt,
    max_new_tokens=128,
    do_sample=False,
    temperature=0.7,
    top_p=0.9,
    use_cache=True,
    return_full_text=True,
):
    result = generate_answers_batch(
        [prompt],
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        temperature=temperature,
        top_p=top_p,
        use_cache=use_cache,
        return_full_text=return_full_text,
    )
    return result[0] if result else ""

def generate_answers_batch(
    prompts,
    max_new_tokens=128,
    do_sample=False,
    temperature=0.7,
    top_p=0.9,
    use_cache=True,
    return_full_text=True,
):
    prompts = [_coerce_to_text(p) for p in prompts]
    if not prompts:
        return []

    # Reserve room for generated tokens so total length stays within model max length.
    input_max_tokens = max(512, MODEL_MAX_SEQ_LEN - max_new_tokens - 8)

    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=input_max_tokens,
    ).to(DEVICE)

    generation_kwargs = dict(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        use_cache=use_cache,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    if do_sample:
        generation_kwargs.update({"temperature": temperature, "top_p": top_p})

    with torch.inference_mode():
        try:
            outputs = model.generate(**generation_kwargs)
        except RuntimeError as err:
            # Fallback for occasional cache-shape issues in some Unsloth/transformers combos.
            if "broadcast shape" in str(err) or "doesn't match" in str(err):
                generation_kwargs["use_cache"] = False
                outputs = model.generate(**generation_kwargs)
            else:
                raise

    decoded_batch = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    if return_full_text:
        return decoded_batch

    trimmed = []
    for prompt, decoded in zip(prompts, decoded_batch):
        trimmed.append(decoded[len(prompt):].strip() if decoded.startswith(prompt) else decoded)
    return trimmed

def extract_answer(text):
    text = _coerce_to_text(text)
    match = re.search(r"Answer:\s*(.*)", text)
    return match.group(1).strip() if match else text.strip()

ex = val_set[0]

prompt = prompt_baseline(ex["question"], build_context(ex))
print(extract_answer(generate_answer(prompt)))
print(prompt)
print("GT:", ex["answer"])

Both `max_new_tokens` (=128) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/

No, they were not.
Answer the question based on the context.

Context:
Ed Wood (film): Ed Wood is a 1994 American biographical period comedy-drama film directed and produced by Tim Burton, and starring Johnny Depp as cult filmmaker Ed Wood.
Ed Wood (film):  The film concerns the period in Wood's life when he made his best-known films as well as his relationship with actor Bela Lugosi, played by Martin Landau.
Ed Wood (film):  Sarah Jessica Parker, Patricia Arquette, Jeffrey Jones, Lisa Marie, and Bill Murray are among the supporting cast.
Scott Derrickson: Scott Derrickson (born July 16, 1966) is an American director, screenwriter and producer.
Scott Derrickson:  He lives in Los Angeles, California.
Scott Derrickson:  He is best known for directing horror films such as "Sinister", "The Exorcism of Emily Rose", and "Deliver Us From Evil", as well as the 2016 Marvel Cinematic Universe installment, "Doctor Strange."
Woodson, Arkansas: Woodson is a census-designated place (CDP) in Pulaski 

In [8]:
def self_consistency(prompt, n=5):
    answers = []
    for _ in range(n):
        out = generate_answer(prompt, do_sample=True, return_full_text=False)
        answers.append(extract_answer(out))
    return Counter(answers).most_common(1)[0][0]

In [9]:
def build_corpus(examples, show_progress=True):
    corpus = []
    iterator = tqdm(examples, desc="Building retrieval corpus") if show_progress else examples
    for ex in iterator:
        text = build_context(ex)
        if text and text.strip():
            corpus.append(text)
    return corpus

corpus = build_corpus(val_set)
if not corpus:
    raise ValueError("Corpus is empty after context extraction. Check dataset context format.")

vectorizer = TfidfVectorizer(token_pattern=r"(?u)\b\w+\b").fit(corpus)
vectors = vectorizer.transform(corpus)

def retrieve(query, k=5):
    q_vec = vectorizer.transform([query])
    scores = cosine_similarity(q_vec, vectors)[0]
    topk = scores.argsort()[-k:][::-1]
    return "\n".join([corpus[i] for i in topk])

def _example_sentence_candidates(example):
    titles, sent_lists = _extract_titles_and_sentences(example)
    candidates = []
    for title, sents in zip(titles, sent_lists):
        for sid, sent in enumerate(sents):
            sent_text = _coerce_to_text(sent).strip()
            if sent_text:
                candidates.append((f"{_coerce_to_text(title)}: {sent_text}", _coerce_to_text(title), sid))
    return candidates

def retrieve_within_example(example, query, k=5):
    candidates = _example_sentence_candidates(example)
    if not candidates:
        return ""

    texts = [c[0] for c in candidates]
    local_vectorizer = TfidfVectorizer(token_pattern=r"(?u)\b\w+\b")
    local_vectors = local_vectorizer.fit_transform(texts)
    q_vec = local_vectorizer.transform([_coerce_to_text(query)])
    scores = cosine_similarity(q_vec, local_vectors)[0]
    topk_idx = scores.argsort()[-k:][::-1]
    return "\n".join(texts[i] for i in topk_idx if scores[i] > 0)

Building retrieval corpus:   0%|          | 0/200 [00:00<?, ?it/s]

In [10]:
from datasets import Dataset
from transformers.pipelines.pt_utils import KeyDataset

NLI_DEVICE = 0 if torch.cuda.is_available() else -1
nli = pipeline("text-classification", model="roberta-large-mnli", top_k=None, device=NLI_DEVICE)

def _truncate_context_for_nli(context, max_chars=2200):
    # RoBERTa MNLI has a strict sequence limit; keep most recent context slice.
    if len(context) <= max_chars:
        return context
    return context[-max_chars:]

def _normalize_nli_scores(output_item):
    if isinstance(output_item, list):
        scores = output_item
    else:
        scores = [output_item]
    return {item["label"].upper(): item["score"] for item in scores}

def _batch_nli_score_maps(context, sentences, batch_size=16):
    clean = [_coerce_to_text(s).strip() for s in sentences if _coerce_to_text(s).strip()]
    if not clean:
        return []

    safe_context = _truncate_context_for_nli(_coerce_to_text(context))

    # Feed the pipeline a Dataset key of plain strings to avoid dict-tokenization errors.
    pair_inputs = [f"{safe_context} </s> {s}" for s in clean]
    pair_dataset = Dataset.from_dict({"inputs": pair_inputs})

    raw_outputs = list(
        nli(
            KeyDataset(pair_dataset, "inputs"),
            truncation=True,
            max_length=512,
            batch_size=batch_size,
        )
    )
    return [_normalize_nli_scores(out) for out in raw_outputs]

def _get_label_prob(score_map, target):
    target = target.upper()
    for label, prob in score_map.items():
        if target in label:
            return prob

    # Fallback for models exposing LABEL_0/LABEL_1/LABEL_2.
    if target == "CONTRADICTION":
        return score_map.get("LABEL_0", 0.0)
    if target == "ENTAILMENT":
        return score_map.get("LABEL_2", 0.0)
    if target == "NEUTRAL":
        return score_map.get("LABEL_1", 0.0)
    return 0.0

def _nli_triplet(score_map):
    return (
        _get_label_prob(score_map, "ENTAILMENT"),
        _get_label_prob(score_map, "CONTRADICTION"),
        _get_label_prob(score_map, "NEUTRAL"),
    )

def unsupported_steps_from_context(context, sentences, entailment_threshold=0.3):
    clean = [_coerce_to_text(s).strip() for s in sentences if _coerce_to_text(s).strip()]
    maps = _batch_nli_score_maps(context, clean)
    unsupported = []

    for sent, scores in zip(clean, maps):
        pred_label = max(scores, key=scores.get)
        entail_prob = _get_label_prob(scores, "ENTAILMENT")
        contradicted = "CONTRADICTION" in pred_label or pred_label == "LABEL_0"
        if contradicted or entail_prob < entailment_threshold:
            unsupported.append(sent)

    return unsupported

def hallucination_and_unsupported(context, sentences, entailment_threshold=0.3):
    maps = _batch_nli_score_maps(context, sentences)
    if not maps:
        return 0, 0

    hallu = 0
    unsup = 0
    for scores in maps:
        pred_label = max(scores, key=scores.get)
        if "CONTRADICTION" in pred_label or pred_label == "LABEL_0":
            hallu = 1
        entail_prob = _get_label_prob(scores, "ENTAILMENT")
        if entail_prob < entailment_threshold:
            unsup = 1
        if hallu and unsup:
            break

    return hallu, unsup

def hallucination_rate(context, sentences):
    hallu, _ = hallucination_and_unsupported(context, sentences)
    return hallu

def unsupported_rate(context, sentences, entailment_threshold=0.3):
    _, unsup = hallucination_and_unsupported(context, sentences, entailment_threshold)
    return unsup

config.json:   0%|          | 0.00/688 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.43G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-large-mnli
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [11]:
def self_verify(q, c, draft, max_rounds=2, entailment_threshold=0.3):
    revised = draft

    for _ in range(max_rounds):
        steps = [s.strip() for s in sent_tokenize(revised) if s.strip()]
        unsupported_steps = unsupported_steps_from_context(
            c, steps, entailment_threshold=entailment_threshold
        )

        if not unsupported_steps:
            break

        flagged = "\n".join([f"- {s}" for s in unsupported_steps[:8]])
        prompt = f"""Revise the reasoning to remove unsupported or contradictory steps.
Use ONLY the provided context.
If the answer is unsupported, answer: NOT ENOUGH INFORMATION.

Context:
{c}

Question: {q}

Current Draft:
{revised}

Flagged Steps:
{flagged}

Return a corrected response in this format:
Reasoning: ...
Answer: ..."""

        revised = generate_answer(prompt, return_full_text=False)

    return revised

In [12]:
import string

def normalize_answer(s):
    def remove_articles(text):
        return re.sub(r'\b(a|an|the)\b', ' ', text)

    def remove_punctuation(text):
        return ''.join(ch for ch in text if ch not in string.punctuation)

    def lowercase(text):
        return text.lower()

    return ' '.join(
        remove_articles(remove_punctuation(lowercase(s))).split()
    )

In [13]:
YES_NO_NOANSWER = {"yes", "no", "noanswer"}

def answer_f1_components(prediction, ground_truth):
    normalized_prediction = normalize_answer(_coerce_to_text(prediction))
    normalized_ground_truth = normalize_answer(_coerce_to_text(ground_truth))

    # Match official HotpotQA behavior for categorical answers.
    if (
        normalized_prediction in YES_NO_NOANSWER
        and normalized_prediction != normalized_ground_truth
    ):
        return 0.0, 0.0, 0.0
    if (
        normalized_ground_truth in YES_NO_NOANSWER
        and normalized_prediction != normalized_ground_truth
    ):
        return 0.0, 0.0, 0.0

    pred_tokens = normalized_prediction.split()
    gold_tokens = normalized_ground_truth.split()

    if len(pred_tokens) == 0 and len(gold_tokens) == 0:
        return 1.0, 1.0, 1.0
    if len(pred_tokens) == 0 or len(gold_tokens) == 0:
        return 0.0, 0.0, 0.0

    common = Counter(pred_tokens) & Counter(gold_tokens)
    num_same = sum(common.values())
    if num_same == 0:
        return 0.0, 0.0, 0.0

    precision = num_same / len(pred_tokens)
    recall = num_same / len(gold_tokens)
    f1 = (2 * precision * recall) / (precision + recall)
    return f1, precision, recall

def exact_match_score(prediction, ground_truth):
    return float(normalize_answer(_coerce_to_text(prediction)) == normalize_answer(_coerce_to_text(ground_truth)))

def score_answer(prediction, gold_answer):
    em = exact_match_score(prediction, gold_answer)
    f1, prec, recall = answer_f1_components(prediction, gold_answer)
    return em, f1, prec, recall

def _gold_sp_pairs(example):
    sf = example.get("supporting_facts", {})

    # HF Hotpot format: {"title": [...], "sent_id": [...]}
    if isinstance(sf, dict):
        titles = sf.get("title", [])
        sent_ids = sf.get("sent_id", [])
        return set((t, int(sid)) for t, sid in zip(titles, sent_ids))

    # Official JSON format: [[title, sent_id], ...].
    if isinstance(sf, list):
        pairs = set()
        for item in sf:
            if isinstance(item, (list, tuple)) and len(item) >= 2:
                pairs.add((_coerce_to_text(item[0]), int(item[1])))
        return pairs

    return set()

def infer_sp_from_text(example, generated_text):
    titles, sent_lists = _extract_titles_and_sentences(example)
    generated_norm = normalize_answer(_coerce_to_text(generated_text))
    pred_pairs = set()

    # Exact sentence-string match against normalized generated text.
    for title, sents in zip(titles, sent_lists):
        for sid, sent in enumerate(sents):
            sent_norm = normalize_answer(_coerce_to_text(sent))
            if sent_norm and sent_norm in generated_norm:
                pred_pairs.add((_coerce_to_text(title), sid))

    return pred_pairs

def _token_set(text):
    return set(normalize_answer(_coerce_to_text(text)).split())

def _sp_overlap_score(sentence_text, query_tokens):
    sent_tokens = _token_set(sentence_text)
    if not sent_tokens or not query_tokens:
        return 0.0
    overlap = len(sent_tokens & query_tokens)
    if overlap == 0:
        return 0.0

    # Balanced coverage: reward overlap from both sentence and query sides.
    return (overlap / len(sent_tokens)) + (overlap / len(query_tokens))

def _candidate_sp_pairs(example):
    titles, sent_lists = _extract_titles_and_sentences(example)
    candidates = []
    for title, sents in zip(titles, sent_lists):
        title_text = _coerce_to_text(title)
        for sid, sent in enumerate(sents):
            sent_text = _coerce_to_text(sent).strip()
            if sent_text:
                full_text = f"{title_text}: {sent_text}"
                candidates.append((title_text, sid, sent_text, full_text))
    return candidates

def _candidate_lookup_by_full_text(candidates):
    lookup = {}
    for title, sid, sent_text, full_text in candidates:
        norm_full = normalize_answer(full_text)
        if norm_full and norm_full not in lookup:
            lookup[norm_full] = (title, sid)
    return lookup

def predict_supporting_facts(example, question, generated_text, max_facts=2):
    generated_text = _coerce_to_text(generated_text)
    answer_text = extract_answer(generated_text)

    # High-precision seed if the model copied evidence text verbatim.
    selected = set(infer_sp_from_text(example, generated_text))

    # Retrieval-guided seed: rank within the current example and map back to title/sent_id.
    candidates = _candidate_sp_pairs(example)
    if candidates and len(selected) < max_facts:
        lookup = _candidate_lookup_by_full_text(candidates)
        retrieval_query = f"{_coerce_to_text(question)} {_coerce_to_text(answer_text)}"
        retrieved_block = retrieve_within_example(example, retrieval_query, k=max(6, max_facts * 3))
        for line in [_coerce_to_text(x).strip() for x in _coerce_to_text(retrieved_block).split("\n") if _coerce_to_text(x).strip()]:
            pair = lookup.get(normalize_answer(line))
            if pair is not None:
                selected.add(pair)
            if len(selected) >= max_facts:
                break

    # Fallback lexical scorer across all candidate sentences.
    if len(selected) < max_facts:
        query_tokens = _token_set(f"{_coerce_to_text(question)} {answer_text} {generated_text}")
        generated_norm = normalize_answer(generated_text)

        scored = []
        for title, sid, sent_text, _ in candidates:
            title_norm = normalize_answer(title)
            title_bonus = 0.15 if title_norm and title_norm in generated_norm else 0.0
            score = _sp_overlap_score(sent_text, query_tokens) + title_bonus
            if score > 0:
                scored.append((score, title, sid))

        scored.sort(key=lambda x: x[0], reverse=True)
        for _, title, sid in scored:
            selected.add((title, sid))
            if len(selected) >= max_facts:
                break

    return selected

def score_supporting_facts(prediction_sp, gold_sp):
    pred_sp = set((
        _coerce_to_text(t),
        int(sid)
    ) for t, sid in prediction_sp)
    gold_sp = set((
        _coerce_to_text(t),
        int(sid)
    ) for t, sid in gold_sp)

    tp = len(pred_sp & gold_sp)
    fp = len(pred_sp - gold_sp)
    fn = len(gold_sp - pred_sp)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
    em = 1.0 if (fp + fn) == 0 else 0.0
    return em, f1, precision, recall

In [14]:
def _chunked(items, batch_size):
    for i in range(0, len(items), batch_size):
        yield items[i : i + batch_size]


def _example_id(example, fallback_index=None):
    raw_id = _coerce_to_text(example.get("_id", example.get("id", ""))).strip()
    if raw_id:
        return raw_id

    # Guard against accidental key collisions if source IDs are missing.
    if fallback_index is not None:
        return f"row-{int(fallback_index)}"

    q = _coerce_to_text(example.get("question", "")).strip()
    a = _coerce_to_text(example.get("answer", "")).strip()
    return f"hash-{abs(hash((q, a)))}"


def _build_batch_contexts(batch, questions, mode, k):
    contexts = []
    for ex, q in zip(batch, questions):
        if mode in ("rag", "rag_cot"):
            # RAG over the current example context is more stable for Hotpot distractor split.
            local_ctx = _coerce_to_text(retrieve_within_example(ex, q, k))
            if local_ctx.strip():
                contexts.append(local_ctx)
            else:
                contexts.append(_coerce_to_text(build_context(ex)))
        else:
            contexts.append(_coerce_to_text(build_context(ex)))
    return contexts


def _build_prompts(mode, questions, contexts):
    prompt_builder = {
        "baseline": prompt_baseline,
        "grounded": prompt_grounded,
        "cot": prompt_cot,
        "cot_cite": prompt_cot_citation,
        "rag": prompt_baseline,
        "rag_cot": prompt_cot,
    }.get(mode)

    if prompt_builder is None:
        raise ValueError(f"Unknown mode: {mode}")

    return [prompt_builder(q, c) for q, c in zip(questions, contexts)]


def _generate_predictions_for_batch(mode, questions, contexts):
    if mode not in ("cot_sc", "verify"):
        prompts = _build_prompts(mode, questions, contexts)
        return generate_answers_batch(prompts, return_full_text=False)

    preds = []
    for q, c in zip(questions, contexts):
        if mode == "cot_sc":
            pred = self_consistency(prompt_cot(q, c))
        else:
            draft = generate_answer(prompt_cot(q, c), return_full_text=False)
            pred = self_verify(q, c, draft)
        preds.append(pred)
    return preds


def _collect_prediction_entries(batch, contexts, preds, prediction, start_index=0):
    hallu, unsup = 0, 0
    entail_sum, contra_sum, neutral_sum = 0.0, 0.0, 0.0
    nli_count = 0

    for offset, (ex, c, pred) in enumerate(zip(batch, contexts, preds)):
        cur_id = _example_id(ex, fallback_index=start_index + offset)
        question = _coerce_to_text(ex.get("question", ""))
        pred_ans = extract_answer(pred)
        pred_sp_pairs = predict_supporting_facts(ex, question, pred, max_facts=2)

        # Official evaluator expects JSON-like lists for supporting facts.
        prediction["answer"][cur_id] = pred_ans
        prediction["sp"][cur_id] = [[t, sid] for t, sid in sorted(pred_sp_pairs)]

        sents = sent_tokenize(_coerce_to_text(pred))
        h, u = hallucination_and_unsupported(c, sents)
        hallu += h
        unsup += u

        maps = _batch_nli_score_maps(c, sents)
        for score_map in maps:
            e, ct, n = _nli_triplet(score_map)
            entail_sum += e
            contra_sum += ct
            neutral_sum += n
            nli_count += 1

    return hallu, unsup, entail_sum, contra_sum, neutral_sum, nli_count


def evaluate(dataset, mode="baseline", k=5, show_progress=True, batch_size=8):
    """
    Generate predictions and score them with official-style HotpotQA logic.
    Additional hallucination/unsupported and NLI-label metrics are reported separately.
    """
    items = list(dataset)
    total = len(items)
    if total == 0:
        return {
            "EM": 0.0,
            "F1": 0.0,
            "SupportingFactEM": 0.0,
            "SupportingFactF1": 0.0,
            "JointEM": 0.0,
            "JointF1": 0.0,
            "Hallucination": 0.0,
            "Unsupported": 0.0,
            "Entailment": 0.0,
            "Contradiction": 0.0,
            "Neutral": 0.0,
        }

    prediction = {"answer": {}, "sp": {}}
    hallu_total, unsup_total = 0, 0
    entail_total, contra_total, neutral_total = 0.0, 0.0, 0.0
    nli_total = 0

    iterator = _chunked(items, batch_size)
    if show_progress:
        total_batches = (total + batch_size - 1) // batch_size
        iterator = tqdm(
            iterator,
            total=total_batches,
            desc=f"Evaluating {mode} (k={k}, bs={batch_size})",
        )

    processed = 0
    for batch in iterator:
        questions = [_coerce_to_text(ex.get("question", "")) for ex in batch]
        contexts = _build_batch_contexts(batch, questions, mode, k)
        preds = _generate_predictions_for_batch(mode, questions, contexts)

        hallu, unsup, entail_sum, contra_sum, neutral_sum, nli_count = _collect_prediction_entries(
            batch,
            contexts,
            preds,
            prediction,
            start_index=processed,
        )
        hallu_total += hallu
        unsup_total += unsup
        entail_total += entail_sum
        contra_total += contra_sum
        neutral_total += neutral_sum
        nli_total += nli_count
        processed += len(batch)

    official = evaluate_official_predictions(prediction, items)
    return {
        "EM": official["EM"],
        "F1": official["F1"],
        "SupportingFactEM": official["SupportingFactEM"],
        "SupportingFactF1": official["SupportingFactF1"],
        "JointEM": official["JointEM"],
        "JointF1": official["JointF1"],
        "Hallucination": hallu_total / total,
        "Unsupported": unsup_total / total,
        "Entailment": (entail_total / nli_total) if nli_total else 0.0,
        "Contradiction": (contra_total / nli_total) if nli_total else 0.0,
        "Neutral": (neutral_total / nli_total) if nli_total else 0.0,
    }


def _empty_hotpot_metrics():
    return {
        "em": 0.0,
        "f1": 0.0,
        "prec": 0.0,
        "recall": 0.0,
        "sp_em": 0.0,
        "sp_f1": 0.0,
        "sp_prec": 0.0,
        "sp_recall": 0.0,
        "joint_em": 0.0,
        "joint_f1": 0.0,
        "joint_prec": 0.0,
        "joint_recall": 0.0,
    }


def _accumulate_answer_metrics(metrics, prediction_text, gold_answer):
    ans_em, ans_f1, ans_prec, ans_recall = score_answer(prediction_text, gold_answer)
    metrics["em"] += ans_em
    metrics["f1"] += ans_f1
    metrics["prec"] += ans_prec
    metrics["recall"] += ans_recall
    return ans_em, ans_prec, ans_recall


def _accumulate_sp_metrics(metrics, prediction_sp, gold_sp):
    sp_em, sp_f1, sp_prec, sp_recall = score_supporting_facts(prediction_sp, gold_sp)
    metrics["sp_em"] += sp_em
    metrics["sp_f1"] += sp_f1
    metrics["sp_prec"] += sp_prec
    metrics["sp_recall"] += sp_recall
    return sp_em, sp_prec, sp_recall


def _accumulate_joint_metrics(metrics, ans_em, ans_prec, ans_recall, sp_em, sp_prec, sp_recall):
    joint_prec = ans_prec * sp_prec
    joint_recall = ans_recall * sp_recall
    joint_f1 = (2 * joint_prec * joint_recall / (joint_prec + joint_recall)) if (joint_prec + joint_recall) > 0 else 0.0
    joint_em = ans_em * sp_em

    metrics["joint_em"] += joint_em
    metrics["joint_f1"] += joint_f1
    metrics["joint_prec"] += joint_prec
    metrics["joint_recall"] += joint_recall


def _normalize_hotpot_metrics(metrics, n):
    for key in metrics:
        metrics[key] /= n


def evaluate_official_predictions(prediction, dataset):
    """
    prediction format:
    {
      "answer": {id: "...", ...},
      "sp": {id: [[title, sent_id], ...], ...}
    }
    """
    answer_pred = prediction.get("answer", {})
    sp_pred = prediction.get("sp", {})

    items = list(dataset)
    n = len(items)
    metrics = _empty_hotpot_metrics()
    if n == 0:
        return metrics

    for idx, ex in enumerate(items):
        cur_id = _example_id(ex, fallback_index=idx)
        can_eval_joint = True

        if cur_id not in answer_pred:
            print(f"missing answer {cur_id}")
            can_eval_joint = False
            ans_em = ans_prec = ans_recall = 0.0
        else:
            ans_em, ans_prec, ans_recall = _accumulate_answer_metrics(
                metrics,
                answer_pred[cur_id],
                ex.get("answer", ""),
            )

        if cur_id not in sp_pred:
            print(f"missing sp fact {cur_id}")
            can_eval_joint = False
            sp_em = sp_prec = sp_recall = 0.0
        else:
            sp_em, sp_prec, sp_recall = _accumulate_sp_metrics(
                metrics,
                sp_pred[cur_id],
                _gold_sp_pairs(ex),
            )

        if can_eval_joint:
            _accumulate_joint_metrics(
                metrics,
                ans_em,
                ans_prec,
                ans_recall,
                sp_em,
                sp_prec,
                sp_recall,
            )

    _normalize_hotpot_metrics(metrics, n)
    return {
        "EM": metrics["em"],
        "F1": metrics["f1"],
        "Precision": metrics["prec"],
        "Recall": metrics["recall"],
        "SupportingFactEM": metrics["sp_em"],
        "SupportingFactF1": metrics["sp_f1"],
        "SupportingFactPrecision": metrics["sp_prec"],
        "SupportingFactRecall": metrics["sp_recall"],
        "JointEM": metrics["joint_em"],
        "JointF1": metrics["joint_f1"],
        "JointPrecision": metrics["joint_prec"],
        "JointRecall": metrics["joint_recall"],
    }

In [15]:
print("Baseline:", evaluate(val_set, "baseline", batch_size=4))  # smaller batch for baseline due to longer prompts


Evaluating baseline (k=5, bs=4):   0%|          | 0/50 [00:00<?, ?it/s]

Both `max_new_tokens` (=128) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:202: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12

Baseline: {'EM': 0.3, 'F1': 0.4158549754196039, 'SupportingFactEM': 0.11, 'SupportingFactF1': 0.3730163568901934, 'JointEM': 0.05, 'JointF1': 0.15655298484228322, 'Hallucination': 0.14, 'Unsupported': 0.98, 'Entailment': 0.03973437265221642, 'Contradiction': 0.15124577287322694, 'Neutral': 0.8090198537136604}


In [ ]:
print("Grounded:", evaluate(val_set, "grounded", batch_size=4))  # smaller batch for grounded due to longer prompts

Evaluating grounded (k=5, bs=4):   0%|          | 0/50 [00:00<?, ?it/s]

Both `max_new_tokens` (=128) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

Grounded: {'EM': 0.25, 'F1': 0.34425832280503166, 'SupportingFactEM': 0.045, 'SupportingFactF1': 0.25368302355686007, 'JointEM': 0.005, 'JointF1': 0.08367733848241282, 'Hallucination': 0.17, 'Unsupported': 0.985}


In [ ]:
print("CoT:", evaluate(val_set, "cot", batch_size=8))

Evaluating cot (k=5, bs=8):   0%|          | 0/25 [00:00<?, ?it/s]

Both `max_new_tokens` (=128) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

CoT: {'EM': 0.185, 'F1': 0.2844466687394444, 'SupportingFactEM': 0.08, 'SupportingFactF1': 0.3285401664140029, 'JointEM': 0.015, 'JointF1': 0.08359750261133488, 'Hallucination': 0.145, 'Unsupported': 0.99}


In [ ]:
print("CoT + SC:", evaluate(val_set, "cot_sc", batch_size=8))

Evaluating cot_sc (k=5, bs=8):   0%|          | 0/25 [00:00<?, ?it/s]

Both `max_new_tokens` (=128) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

CoT + SC: {'EM': 0.205, 'F1': 0.28256729994391705, 'SupportingFactEM': 0.09, 'SupportingFactF1': 0.40454761904761893, 'JointEM': 0.02, 'JointF1': 0.13241142008751086, 'Hallucination': 0.115, 'Unsupported': 0.94}


In [ ]:
print("CoT + Citation:", evaluate(val_set, "cot_cite", batch_size=8))

Evaluating cot_cite (k=5, bs=8):   0%|          | 0/25 [00:00<?, ?it/s]

Both `max_new_tokens` (=128) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

CoT + Citation: {'EM': 0.12, 'F1': 0.21930305089208577, 'SupportingFactEM': 0.06, 'SupportingFactF1': 0.3059727060965426, 'JointEM': 0.005, 'JointF1': 0.045633382553903076, 'Hallucination': 0.14, 'Unsupported': 0.99}


In [ ]:
print("RAG (k=2):", evaluate(val_set, "rag", k=2, batch_size=8))

Evaluating rag (k=2, bs=8):   0%|          | 0/25 [00:00<?, ?it/s]

Both `max_new_tokens` (=128) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

RAG (k=2): {'EM': 0.25, 'F1': 0.33297972982310925, 'SupportingFactEM': 0.105, 'SupportingFactF1': 0.44045238095238104, 'JointEM': 0.055, 'JointF1': 0.18705291645043817, 'Hallucination': 0.185, 'Unsupported': 0.455}


In [ ]:
print("RAG (k=5):", evaluate(val_set, "rag", k=5, batch_size=8))

Evaluating rag (k=5, bs=8):   0%|          | 0/25 [00:00<?, ?it/s]

Both `max_new_tokens` (=128) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

RAG (k=5): {'EM': 0.19, 'F1': 0.27493469211009425, 'SupportingFactEM': 0.09, 'SupportingFactF1': 0.43617460317460327, 'JointEM': 0.025, 'JointF1': 0.1469424398726781, 'Hallucination': 0.34, 'Unsupported': 0.625}


In [ ]:
print("RAG (k=10):", evaluate(val_set, "rag", k=10, batch_size=8))

Evaluating rag (k=10, bs=8):   0%|          | 0/25 [00:00<?, ?it/s]

Both `max_new_tokens` (=128) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

RAG (k=10): {'EM': 0.24, 'F1': 0.35909726871789305, 'SupportingFactEM': 0.095, 'SupportingFactF1': 0.4143676323676322, 'JointEM': 0.045, 'JointF1': 0.18297162162095454, 'Hallucination': 0.41, 'Unsupported': 0.855}


In [ ]:
print("RAG + CoT (k=2):", evaluate(val_set, "rag_cot", k=2, batch_size=8))

Evaluating rag_cot (k=2, bs=8):   0%|          | 0/25 [00:00<?, ?it/s]

Both `max_new_tokens` (=128) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

RAG + CoT (k=2): {'EM': 0.045, 'F1': 0.15122679279411547, 'SupportingFactEM': 0.12, 'SupportingFactF1': 0.474952380952381, 'JointEM': 0.015, 'JointF1': 0.08843759206744745, 'Hallucination': 0.3, 'Unsupported': 0.6}


In [ ]:
print("RAG + CoT (k=5):", evaluate(val_set, "rag_cot", k=5, batch_size=8))

Evaluating rag_cot (k=5, bs=8):   0%|          | 0/25 [00:00<?, ?it/s]

Both `max_new_tokens` (=128) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

RAG + CoT (k=5): {'EM': 0.135, 'F1': 0.2451995739948683, 'SupportingFactEM': 0.115, 'SupportingFactF1': 0.4397222222222224, 'JointEM': 0.01, 'JointF1': 0.12826754538915652, 'Hallucination': 0.375, 'Unsupported': 0.67}


In [ ]:
print("RAG + CoT (k=10):", evaluate(val_set, "rag_cot", k=10, batch_size=8))

Evaluating rag_cot (k=10, bs=8):   0%|          | 0/25 [00:00<?, ?it/s]

Both `max_new_tokens` (=128) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

RAG + CoT (k=10): {'EM': 0.1, 'F1': 0.23186810724025356, 'SupportingFactEM': 0.14, 'SupportingFactF1': 0.45603429903429893, 'JointEM': 0.015, 'JointF1': 0.11536214429407243, 'Hallucination': 0.46, 'Unsupported': 0.92}


In [ ]:
print("Self-Verify:", evaluate(val_set, "verify", batch_size=8))

Evaluating verify (k=5, bs=8):   0%|          | 0/25 [00:00<?, ?it/s]

Both `max_new_tokens` (=128) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

Self-Verify: {'EM': 0.01, 'F1': 0.011695210429211842, 'SupportingFactEM': 0.0, 'SupportingFactF1': 0.11877376271125334, 'JointEM': 0.0, 'JointF1': 0.0013016177146926114, 'Hallucination': 0.155, 'Unsupported': 0.99}
